In [ ]:
import pandas as pd
from pathlib import Path
import sys
from layoff.config import MERGED_DATA, DATA_PROCESSED, LABELED

def statement_to_dict_df(file, statement_name):
    ticker = Path(file).stem.replace(f"_{statement_name}", "")

    df = pd.read_csv(file)

    rows = []

    # First column contains metric names
    metric_col = df.columns[0]

    # Remaining columns are dates
    for date in df.columns[1:]:
        metrics = (
            df[[metric_col, date]]
            .dropna()
            .set_index(metric_col)[date]
            .to_dict()
        )

        rows.append({
            "company": ticker,
            "date": date,
            statement_name: metrics
        })

    return pd.DataFrame(rows)

In [2]:
from pathlib import Path
from layoff.config import DATA_DIR, DATA_PROCESSED, DATA_BALANCE_SHEET, DATA_FINANCIALS

DATA_TICKER = DATA_PROCESSED/ "layoffs_with_tickers.csv"
tickers = set()

for file in DATA_BALANCE_SHEET.glob("*.csv"):
    ticker = file.stem.replace("_balancesheet", "")
    tickers.add(ticker)
 
print(f"Found {len(tickers)} companies")

Found 2708 companies


In [3]:
master_rows = []

for ticker in tickers:

    dfs = []

    statement_info = [
        ("balancesheet", DATA_BALANCE_SHEET, "_balancesheet.csv"),
        ("cashflow", DATA_BALANCE_SHEET, "_cashflow.csv"),
        ("financials", DATA_FINANCIALS, "_financials.csv"),
    ]

    for statement_name, directory, suffix in statement_info:

        file = directory / f"{ticker}{suffix}"

        if not file.exists():
            continue

        try:
            statement_df = statement_to_dict_df(
                file,
                statement_name
            )

            if not statement_df.empty:
                dfs.append(statement_df)

        except Exception as e:
            print(
                f"Failed {statement_name} "
                f"for {ticker}: {e}"
            )

    # Skip companies with no statements
    if len(dfs) == 0:
        continue

    # Merge all available statements
    company_df = dfs[0]

    for df in dfs[1:]:
        company_df = company_df.merge(
            df,
            on=["company", "date"],
            how="outer"
        )

    master_rows.append(company_df)

    print(f"Processed {ticker}")

Processed BOLT
Processed WLK
Processed KVUE
Processed K9B.F
Processed TGT
Processed AVY
Processed DGCU2.BA
Processed NVS
Processed 2305.T
Processed MMM
Processed SUNPHARMA.NS
Processed 601012.SS
Processed 0216.KL
Processed KRMN
Processed EQX.TO
Processed STNE
Processed BAVA.CO
Processed ECP.F
Processed 8566.T
Processed ZBRA
Processed VREX
Processed SBSW
Processed CALY
Processed MS
Processed 2689.HK
Processed BHARATAGRI.BO
Processed DDD
Processed 4977.T
Processed EQ
Processed WISH.V
Processed 002623.SZ
Processed 6501.T
Processed AGYS
Processed ARM
Processed CXM
Processed TTWO
Processed 8402.HK
Processed FAURY
Processed VLTSAP.XC
Processed HFG.F
Processed TANAA.BO
Processed DK
Processed 3026.KL
Processed VRM
Processed PARAS.NS
Processed T06.F
Processed FVRR
Processed RAIN
Processed PZZA
Processed WEST
Processed CMG
Processed QCOM
Processed RJ1.F
Processed NWL
Processed TAC
Processed WOLF
Processed SCPX
Processed KHC
Processed COHU
Processed 2339.HK
Processed IQE.L
Processed AMBP
Processe

In [4]:
import pandas as pd

dataset = pd.concat(master_rows, ignore_index=True)
dataset["date"] = pd.to_datetime(dataset["date"])

dataset["quarter"] = dataset["date"].dt.to_period("Q").astype(str)

bs_features = pd.json_normalize(
    dataset["balancesheet"]
).add_prefix("bs_")


cf_features = pd.json_normalize(
    dataset["cashflow"]
).add_prefix("cf_")

fin_features = pd.json_normalize(
    dataset["financials"]
).add_prefix("fin_")

features_df = pd.concat(
    [
        dataset[["company", "date", "quarter"]],
        bs_features,
        cf_features,
        fin_features,
    ],
    axis=1,
)
# features_df.fillna("NaN")
# print(features_df.fillna("NaN").head())

features_df.to_csv(
    MERGED_DATA["MERGED_OUTPUT_CSV_PATH"],
    index=False
)

# Verify
print(features_df.shape)
print(features_df.head())

KeyError: 'cashflow'